# Project FORESIGHT — 03: Baseline Forecasting, Model Training & Backtesting

**Objective**: Benchmark 5 classical forecasting baselines, train candidate supervised machine learning models, execute walk-forward rolling-origin backtesting across 12 origins and 8 horizons, and validate the selected production Hybrid model architecture.

**Key Architecture Components**:
- **Evaluation Protocol**: Walk-forward rolling origins (12 folds, 52 weeks min training, $h=1..8$ weeks).
- **Primary Metric**: Volume-Weighted Absolute Percentage Error (WAPE).
- **Candidate Models**: Naive, Seasonal Naive, Moving Average (MA4, MA8), SES, LightGBM, Random Forest, XGBoost.
- **Production Selection**: Hybrid Architecture ($h=1$ Random Forest, $h=2$ XGBoost, $h=3..8$ Seasonal Naive).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CFG, PATHS
from src.baseline import (
    load_analysis_ready_data,
    aggregate_weekly_demand,
    build_rolling_origins,
    generate_baseline_forecasts,
    compute_overall_metrics,
    compute_macro_metrics,
    compute_metrics_by_horizon,
    compute_metrics_by_category,
    compute_metrics_by_sku,
    verify_data_integrity,
)

print(f"FORESIGHT Baseline & Model Evaluation Framework Initialized. Seed: {CFG.random_seed}")

## 1. Aggregate Weekly Demand & Volume Conservation Check

In [ ]:
daily_df = load_analysis_ready_data()
weekly_df = aggregate_weekly_demand(daily_df)
print(f"Daily Shape : {daily_df.shape} (36,550 rows, 50 SKUs x 731 dates)")
print(f"Weekly Shape: {weekly_df.shape} (50 SKUs x 106 weeks)")
print(f"Daily Total Units : {daily_df['Units_Sold'].sum():,}")
print(f"Weekly Total Units: {weekly_df['Units_Sold'].sum():,}")
assert daily_df["Units_Sold"].sum() == weekly_df["Units_Sold"].sum(), "Volume reconciliation failed!"
print("Volume Conservation: VERIFIED [OK]")

## 2. Rolling-Origin Backtesting Configuration

Walk-forward backtesting guarantees zero temporal leakage:
- 12 rolling folds stepping ~3-4 weeks across the validation period.
- Minimum training window: 52 weeks (1 full year of seasonal history).
- Forecast horizon: 8 weeks ($h=1..8$).

In [ ]:
origins = build_rolling_origins(weekly_df, n_origins=12, horizon=8, min_train_weeks=52)
print(f"Generated {len(origins)} rolling origins:")
for i, o in enumerate(origins, 1):
    print(f"  Fold {i:02d}: Origin Date = {o.strftime('%Y-%m-%d')}")

## 3. Classical Baseline Forecasting Evaluation

Evaluate 5 classical methods: Naive, Seasonal Naive (lag 52), MA4, MA8, SES ($\alpha=0.3$).

In [ ]:
forecasts_df = generate_baseline_forecasts(weekly_df, origins, horizon=8)
print(f"Generated {len(forecasts_df):,} baseline forecast records.")
summary_df = compute_overall_metrics(forecasts_df)
macro_df = compute_macro_metrics(forecasts_df)

print("=== OVERALL (VOLUME-WEIGHTED) BASELINE WAPE ===")
print(summary_df.to_string(index=False))
print("\n=== MACRO (UNWEIGHTED SKU AVERAGE) BASELINE WAPE ===")
print(macro_df.to_string(index=False))

## 4. Baseline Performance by Forecast Horizon ($h=1..8$)

In [ ]:
horizon_df = compute_metrics_by_horizon(forecasts_df)
pivot_wape = horizon_df.pivot(index="horizon", columns="model", values="WAPE").round(2)
print("=== BASELINE WAPE (%) BY HORIZON ===")
print(pivot_wape)

## 5. Candidate Machine Learning Models & Final Evaluation

Load the comprehensive backtest evaluation results comparing classical baselines against tuned ML candidates (LightGBM, Random Forest, XGBoost) and the Selected Hybrid architecture.

In [ ]:
final_metrics_path = PATHS.artifacts_dir / "models" / "final" / "final_by_horizon.csv"
if final_metrics_path.exists():
    final_metrics_df = pd.read_csv(final_metrics_path)
    print("=== MODEL PERFORMANCE BY HORIZON (WAPE %) ===")
    pivot_comp = final_metrics_df.pivot(index="horizon", columns="model", values="WAPE").round(2)
    print(pivot_comp)
else:
    print(f"Note: {final_metrics_path} not found.")

## 6. Production Model Architecture Inspection

Inspect the production model registry and ratified model assignments per horizon:
- $h=1$: Random Forest (captures high-frequency non-linear immediate signals)
- $h=2$: XGBoost (optimizes gradient-boosted medium-lead dynamics)
- $h=3..8$: Seasonal Naive (robust against long-range variance inflation)

In [ ]:
import json
arch_path = PATHS.artifacts_dir / "models" / "final" / "architecture.json"
if arch_path.exists():
    with open(arch_path, "r", encoding="utf-8") as f:
        arch = json.load(f)
    print("=== PRODUCTION ARCHITECTURE CONFIGURATION ===")
    print(f"Model Name        : {arch.get('model_name')}")
    print(f"Selection Strategy: {arch.get('selection_strategy')}")
    print(f"Horizon Models    : {arch.get('horizon_models')}")
    print(f"Mean WAPE         : {arch.get('mean_wape')}%")
else:
    print("Architecture file not found.")

## 7. Model Diagnostic Artifacts & Verification

In [ ]:
plots_dir = PATHS.artifacts_dir / "baseline" / "plots"
if plots_dir.exists():
    plots = sorted(list(plots_dir.glob("*.png")))
    print(f"Found {len(plots)} baseline visualization plots in {plots_dir}:")
    for p in plots[:6]:
        print(f"  - {p.name}")

status = verify_data_integrity()
print(f"\nData Integrity Check: {'PASSED [OK]' if status else 'FAILED'}")